# Compound retrieval and inference in one SQL query

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.52.0/cookbook/notebooks/recipes/compound_query.ipynb)

Built from [`cookbook/recipes/compound_query/example.py`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/recipes/compound_query/example.py). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.52.0 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, on the CPU, in minutes. SCALE = "full"
# runs it over the published data and real encoders instead, on the GPU: the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.52.0", engine + "==0.52.0", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.52.0#subdirectory=cookbook/book", server + "==0.52.0"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

Join sources, filter, and run a model over a relation — in one statement the
engine plans — embedded or against a `jammi-server` over Flight SQL.

**When to use this pattern.** You want results enriched with context from
other sources (who holds a patent, which category it is in) and with model
output computed inside the query, without a per-row round-trip. `search`
returns a bounded top-k; anything caller-shaped rides SQL, where the
`annotate(...)` table function runs a model over a relation.

## What `example.py` does

1. Registers a 20-row patent corpus and a small `assignees` table
2. Enriches by joining: every patent gains its assignee's company and country
3. `annotate('<model>', 'text_embedding', 'corpus.public.tiny_corpus', 'id', 'content')`
   embeds every patent inside the query, joined back to its row and its
   assignee and filtered to US assignees — one statement
4. Runs the identical SQL against a `jammi-server` (`jammi.testing.LiveServer`)
   over the Flight SQL lane and checks the same rows and vectors come back

In [ ]:
from __future__ import annotations

import csv
import tempfile
from pathlib import Path

import jammi
import numpy as np
from jammi.testing import LiveServer
from jammi_cookbook import fixtures

CORPUS_URL = fixtures.url("tiny_corpus.parquet")
MODEL = fixtures.model("tiny_bert")

# Who holds each patent: the corpus's `assignee_id` values, 101–110.
ASSIGNEES = [
    (101, "Aurora Photonics", "US"),
    (102, "Brightwell Bio", "DE"),
    (103, "Cobalt Materials", "JP"),
    (104, "Delta Quantum", "US"),
    (105, "Ember Chemical", "DE"),
    (106, "Fjord Energy", "NO"),
    (107, "Granite Therapeutics", "US"),
    (108, "Harbor Robotics", "JP"),
    (109, "Iris Genomics", "US"),
    (110, "Juniper AI", "US"),
]

ENRICHED = """
    SELECT p.id, p.title, a.company_name, a.country
    FROM corpus.public.tiny_corpus AS p
    JOIN assignees.public.assignees AS a ON p.assignee_id = a.id
    ORDER BY p.id
"""

# Search → join → annotate in one statement: the model runs inside the query.
ANNOTATED = f"""
    SELECT p.id, p.title, a.company_name, ann.vector
    FROM annotate('{MODEL}', 'text_embedding',
                  'corpus.public.tiny_corpus', 'id', 'content') AS ann
    JOIN corpus.public.tiny_corpus  AS p ON ann._row_id = arrow_cast(p.id, 'Utf8')
    JOIN assignees.public.assignees AS a ON p.assignee_id = a.id
    WHERE a.country = 'US'
    ORDER BY p.id
"""


def write_assignees(directory: Path) -> str:
    """The `assignees` table as a CSV file, returned as its `file://` URL."""
    path = directory / "assignees.csv"
    with path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "company_name", "country"])
        writer.writerows(ASSIGNEES)
    return path.as_uri()


def compound(db, assignees_url: str) -> tuple[list[dict], list[dict]]:
    """The same program on either transport: the enriched rows, then the annotated ones."""
    db.add_source("corpus", url=CORPUS_URL, format="parquet")
    db.add_source("assignees", url=assignees_url, format="csv")
    return db.sql(ENRICHED).to_pylist(), db.sql(ANNOTATED).to_pylist()


def main() -> int:
    with tempfile.TemporaryDirectory() as scratch:
        assignees_url = write_assignees(Path(scratch))

        with tempfile.TemporaryDirectory() as local_dir, jammi.connect(f"file://{local_dir}") as db:
            enriched, annotated = compound(db, assignees_url)

        # Every patent gains its assignee's company and country from the join.
        print(f"enriched: {len(enriched)} patents, e.g. {enriched[0]}")
        assert len(enriched) == 20 and all(r["company_name"] for r in enriched)

        # The annotated query keeps exactly the US-held patents — each once —
        # with a vector the model computed from that patent's text.
        us_ids = [r["id"] for r in enriched if r["country"] == "US"]
        assert [r["id"] for r in annotated] == us_ids, "annotate kept every US patent once"
        width = len(annotated[0]["vector"])
        assert width > 0 and all(len(r["vector"]) == width for r in annotated)
        print(f"annotated: {len(annotated)} US-held patents, {width}-dim vectors")
        for row in annotated[:3]:
            print(f"  {row['id']:>2}  {row['company_name']:<20}  {row['title']}")

        # The identical SQL against a server, over the Flight SQL lane.
        with tempfile.TemporaryDirectory() as srv_dir, LiveServer(srv_dir) as server:
            with jammi.connect(server.endpoint) as remote:
                remote_enriched, remote_annotated = compound(remote, assignees_url)

        assert remote_enriched == enriched, "the join is the same on both transports"
        assert [r["id"] for r in remote_annotated] == us_ids
        for local, served in zip(annotated, remote_annotated):
            np.testing.assert_allclose(served["vector"], local["vector"], rtol=1e-5, atol=1e-6)
        print(f"remote ({server.endpoint}): the same {len(remote_annotated)} rows and vectors")

    print("compound_query: OK")
    return 0

In [ ]:
assert main() == 0